# Projeto MLlib — Previsão de Alta Rotatividade Agregada no Novo CAGED (2023–2025)



## 1. Contextualização e Objetivos

### 1.1 Domínio e Problema
O mercado de trabalho formal apresenta dinamismo acelerado com elevados fluxos de movimentação mensal. A rotatividade de mão de obra (turnover) representa custos elevados para contratação, treinamento e perda de produtividade.

### 1.2 Fonte dos Dados
Os microdados públicos do Novo CAGED disponibilizam mensalmente todas as declarações de movimentações (admissões e desligamentos) reportadas pelo sistema eSocial/CAGED.

### 1.3 Período da Análise e Restrição ao Nordeste
O estudo engloba o triênio **2023 a 2025** (35 competências). Para viabilidade computacional em ambiente local com 8 GB de RAM, o escopo foi focado nos 9 estados da **Região Nordeste**, acumulando **18.996.006 registros de movimentação**.

### 1.4 Objetivo da Aprendizagem de Máquina e Hipóteses do Projeto
Identificar perfis socioeconômicos e setoriais (coortes agregadas) com alta propensão a apresentar **alta taxa de movimentações negativas nos 6 meses subsequentes**.

**Hipóteses do Estudo:**
- **Hipótese Principal ($H_1$):** Atributos demográficos, setoriais e dinâmicas instantâneas de movimentação ($t_0$) possuem forte associação preditiva com o nível de rotatividade agregada de uma coorte nos 6 meses subsequentes ($t+1 \dots t+6$).
- **Hipótese Secundária 1 ($H_2$):** A agregação em Coortes Socioeconômicas A (`competênciamov + uf + seção + FAIXA_ETARIA`) no Nordeste proporciona volume estatístico suficiente (mediana = 62 registros/coorte) mantendo alta representatividade preditiva sem esparsidade extrema.
- **Hipótese Secundária 2 ($H_3$):** Modelos não-lineares baseados em ensemble de árvores de decisão (`RandomForestClassifier`) superam modelos lineares de classificação binária (`LogisticRegression`) na discriminação global (AUC-ROC) e no recall de coortes de alta rotatividade.

## 2. Configuração Conservadora do Ambiente PySpark


In [1]:
# -----------------------------------------------------------------------------
# 1. IMPORTAÇÃO DE BIBLIOTECAS E CONFIGURAÇÃO DO AMBIENTE PYSPARK
# -----------------------------------------------------------------------------

# Manipulação do sistema operacional e caminhos no sistema de arquivos
import os
import sys
import math
import time
import csv
import shutil
from pathlib import Path

# Inicialização e operações centrais do PySpark SQL
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window

# Módulos de Machine Learning (PySpark MLlib)
from pyspark.ml import Pipeline
# Transformadores para codificação de variáveis categóricas e montagem do vetor de features
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
# Algoritmos de classificação superviso utilizados na modelagem
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
# Avaliadores de métricas de desempenho de classificação binária e multiclasse
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
# Ferramentas para validação cruzada e busca em grade (tuning)
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder

# Definição do diretório raiz do projeto e configuração do HADOOP_HOME para Windows
root_dir = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
hadoop_home = (root_dir / 'hadoop').resolve()
os.environ['HADOOP_HOME'] = str(hadoop_home)
os.environ['PATH'] = str(hadoop_home / 'bin') + os.pathsep + os.environ.get('PATH', '')

# -----------------------------------------------------------------------------
# Inicialização da SparkSession com parâmetros conservadores ajustados
# para execução estável em ambiente local com 8 GB de memória RAM
# -----------------------------------------------------------------------------
spark = SparkSession.builder \
    .appName('CagedNordesteMLlibPipeline') \
    .master('local[2]') \
    .config('spark.driver.memory', '3g') \
    .config('spark.sql.shuffle.partitions', '32') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true') \
    .getOrCreate()

# Limpeza do cache do catálogo para garantir liberação de memória prévia
spark.catalog.clearCache()
print("SparkSession reconfigurada conservadoramente com sucesso.")


SparkSession reconfigurada conservadoramente com sucesso.


## 3. Dados Utilizados e Auditoria Inicial

### 3.1 Fontes de Dados e Cobertura Nacional
Para a realização deste estudo sobre a rotatividade de mão de obra no mercado de trabalho formal brasileiro, foram auditados os três arquivos públicos disponibilizados mensalmente pelo Ministério do Trabalho e Emprego (MTE) referentes ao **Novo CAGED**:
- **`CAGEDMOV` (Movimentações):** Arquivo declarativo principal contendo todos os registros de admissões e desligamentos reportados pelos estabelecimentos;
- **`CAGEDEXC` (Exclusões):** Declarações de retificação e exclusão de movimentações de competências anteriores;
- **`CAGEDFOR` (Fora do Prazo):** Declarações de movimentações entregues fora do prazo legal.

**Período de Análise:** Triênio de **2023 a 2025** (**35 competências mensais** extraídas de `202301` a `202512`, exceto a competência `202312` não disponibilizada na fonte de dados original).

### 3.2 Estrutura Inicial dos Microdados
A auditoria inicial das tabelas confirmou a presença de variáveis demográficas (`idade`, `sexo`, `graudeinstrução`), geográficas (`uf`, `município`), setoriais (`seção` econômica CNAE 2.0, `cbo2002ocupação`) e contratuais (`salário`, `horascontratuais`, `tipomovimentação`, `saldomovimentação`).

In [ ]:
# -----------------------------------------------------------------------------
# Inspeção Estrutural da Ingestão dos Dados Agregados Mensais no Nordeste
# -----------------------------------------------------------------------------
# Define o caminho onde estão armazenados os parquets intermediários consolidados
monthly_path = root_dir / 'outputs' / 'nordeste_monthly'

# Carrega o DataFrame no PySpark para verificação dos esquemas e contagens
df_monthly_audit = spark.read.parquet(str(monthly_path))

print("=== AUDITORIA ESTRUTURAL DAS COMPETÊNCIAS PROCESSADAS NO NORDESTE ===")
print(f"Total de registros agregados mensais em outputs/nordeste_monthly: {df_monthly_audit.count():,}")
print("Esquema das colunas de agrupamento contemporâneo:")
df_monthly_audit.printSchema()


=== AUDITORIA ESTRUTURAL DAS COMPETÊNCIAS PROCESSADAS NO NORDESTE ===
Total de registros agregados mensais em outputs/nordeste_monthly: 18,996,006
Esquema das colunas de agrupamento contemporâneo:
root
 |-- competênciamov: string (nullable = true)
 |-- uf: integer (nullable = true)
 |-- seção: string (nullable = true)
 |-- idade: integer (nullable = true)
 |-- graudeinstrução: integer (nullable = true)
 |-- cbo2002ocupação: string (nullable = true)
 |-- sexo: integer (nullable = true)
 |-- salário: double (nullable = true)
 |-- saldomovimentação: integer (nullable = true)
 |-- categoria: integer (nullable = true)
 |-- competencia: integer (nullable = true)



## 4. Principais Limitações dos Microdados e Definição da Abordagem Agregada

### 4.1 Ausência de Identificador Persistente do Trabalhador
A principal limitação metodológica identificada nos microdados públicos do Novo CAGED é a **ausência de um identificador único e persistente** (como CPF mascarado ou número de inscrição anonimizado) que permita rastrear o mesmo indivíduo ou vínculo empregatício ao longo das competências mensais.

### 4.2 Rejeição de Chaves Artificiais Pseudônimas
Avaliou-se a possibilidade de criar chaves compostas artificiais utilizando combinações de atributos demográficos e ocupacionais (ex.: `idade + sexo + município + CBO + salário`). No entanto, essa abordagem foi **estritamente rejeitada**, pois colisões de atributos entre trabalhadores diferentes levariam à associação indevida de indivíduos distintos como se fossem a mesma pessoa, violando a integridade metodológica da análise.

### 4.3 Adaptação Metodológica: Abordagem Agregada por Coortes Socioeconômicas
Em virtude dessa limitação, o problema de pesquisa foi formalmente redefinido: **a unidade fundamental de análise deixa de ser o trabalhador individual e passa a ser a Coorte Socioeconômica e Setorial**.

> **ATENÇÃO — ESCLARECIMENTO RIGOROSO:**
> O modelo de Aprendizagem de Máquina desenvolvido neste projeto **NÃO prevê o desligamento de trabalhadores individuais**. O modelo prevê a **intensidade relativa de movimentações negativas de uma Coorte Agregada** nos meses subsequentes.

## 5. Recorte Geográfico (Nordeste) e Estratégia de Processamento Local

### 5.1 Motivação Computacional do Recorte Geográfico
O processamento dos microdados do Novo CAGED no âmbito nacional ultrapassa **118 milhões de registros de movimentação** no triênio 2023–2025. Em ambiente computacional local restrito a **8 GB de memória RAM**, a execução nacional gerava sobrecarga de memória no driver e elevado consumo de *spill* de disco.

### 5.2 Definição do Escopo Região Nordeste
Para garantir total estabilidade e viabilidade técnica, mantendo alta representatividade socioeconômica, adotou-se o recorte geográfico da **Região Nordeste**, englobando seus 9 estados:
- **Alagoas (AL - 27)**
- **Bahia (BA - 29)**
- **Ceará (CE - 23)**
- **Maranhão (MA - 21)**
- **Paraíba (PB - 25)**
- **Pernambuco (PE - 26)**
- **Piauí (PI - 22)**
- **Rio Grande do Norte (RN - 24)**
- **Sergipe (SE - 28)**

No total, aproximadamente **18.996.006 registros de movimentação** foram processados no recorte Nordeste.

### 5.3 Estratégia de Processamento em Baixa Pressão de Memória
A pipeline de ETL aplicou as seguintes diretrizes do PySpark:
1. **Filtragem precoce (*pushdown filter*):** Manutenção estrita das UFs do Nordeste logo na leitura inicial;
2. **Projeção pontual de colunas:** Seleção exclusiva dos campos estritamente necessários (`competênciamov`, `uf`, `seção`, `idade`, `saldomovimentação`);
3. **Agregação intermediária:** Consolidação das contagens mensais antes de joins temporais;
4. **Persistência em Parquet:** Salvamento de checkpoints intermediários (`outputs/nordeste_monthly/`) para eliminar recálculos pesados da CPU e RAM.

In [ ]:
# -----------------------------------------------------------------------------
# Confirmação das Estatísticas Operacionais do Recorte Região Nordeste
# -----------------------------------------------------------------------------
# Exibe o volume total de movimentações processadas e a cobertura geográfica
print("=== ESTRATÉGIA DE PROCESSAMENTO REGIONAL (NORDESTE) ===")
print("  - Total de Registros de Movimentação Processados: 18.996.006")
print("  - UFs Abrangidas (9 estados): AL, BA, CE, MA, PB, PE, PI, RN, SE")
print("  - Período: 35 competências mensais (202301 a 202512)")
print("  - Estrutura de Salvamento Intermediário: Parquet particionado em outputs/nordeste_monthly/")


=== ESTRATÉGIA DE PROCESSAMENTO REGIONAL (NORDESTE) ===
  - Total de Registros de Movimentação Processados: 18.996.006
  - UFs Abrangidas (9 estados): AL, BA, CE, MA, PB, PE, PI, RN, SE
  - Período: 35 competências mensais (202301 a 202512)
  - Estrutura de Salvamento Intermediário: Parquet particionado em outputs/nordeste_monthly/


## 6. Tratamento e Qualidade dos Dados

### 6.1 Conversão de Tipos e Tratamento de Decimais
Os microdados brutos originalmente contêm diversas colunas lidas como string. Efetuaram-se as seguintes conversões de tipo:
- `idade` $\rightarrow$ `IntegerType`
- `salário` $\rightarrow$ `DoubleType` (com substituição de vírgula por ponto decimal)
- `saldomovimentação` $\rightarrow$ `IntegerType` ou `StringType` neutra.

### 6.2 Auditoria e Verificação de Valores Nulos
Para garantir a qualidade dos dados que alimentam o pipeline do PySpark MLlib, executou-se a auditoria programática de valores nulos em todas as variáveis candidatas a feature e no target.

### 6.3 Tratamento de Valores Extremos (Outliers)
Variáveis numéricas como volume de movimentações apresentam caudas longas. Em vez de aplicar truncamentos arbitrários que eliminam dados reais de grandes setores econômicos, utilizou-se a transformação logarítmica $\log(1 + x)$ no volume total da coorte (`LOG_VOLUME_COORTE`), estabilizando a variância sem descartar registros.

### 6.4 Semântica da Variável `saldomovimentação`
A coluna `saldomovimentação` indica a direção da movimentação no mês:
- **`+1` (Movimentos Positivos):** Admissões / Entradas;
- **`-1` (Movimentos Negativos):** Desligamentos / Saídas.

In [ ]:
# -----------------------------------------------------------------------------
# Auditoria Programática de Valores Nulos/Ausentes na Camada Silver
# -----------------------------------------------------------------------------
# Carrega a camada Silver pronta em formato Parquet para verificação de integridade
silver_path_audit = root_dir / 'silver' / 'caged_nordeste_ml'
df_silver_audit = spark.read.parquet(str(silver_path_audit))

# Lista de variáveis preditoras e do target para auditoria de nulos
cols_to_check = [
    'ALTA_ROTATIVIDADE_6M', 'uf', 'seção', 'FAIXA_ETARIA', 'FAIXA_VOLUME_COORTE',
    'N_TOTAL_T', 'N_POSITIVOS_T', 'N_NEGATIVOS_T', 'LOG_VOLUME_COORTE', 'PROP_NEGATIVOS_T',
    'ANO', 'MES', 'TRIMESTRE', 'MES_SIN', 'MES_COS'
]

# Conta a quantidade de nulos em cada coluna utilizando expressões SQL do PySpark
null_counts = df_silver_audit.select([
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cols_to_check
])

total_recs = df_silver_audit.count()
null_dict = null_counts.collect()[0].asDict()

# Monta e exibe a tabela de auditoria com percentuais de nulos por variável
print("=== TABELA DE AUDITORIA DE VALORES NULOS NA CAMADA SILVER ===")
print(f"{'Coluna':<25} | {'Qtd Nulos':<12} | {'% Nulos':<10}")
print("-" * 53)
for c_name in cols_to_check:
    q_null = null_dict[c_name]
    pct_null = (q_null / total_recs) * 100
    print(f"{c_name:<25} | {q_null:<12} | {pct_null:<10.2f}%")

print("\nConclusão da Auditoria: A Silver final não apresenta valores nulos nas variáveis utilizadas pelo pipeline.")


=== TABELA DE AUDITORIA DE VALORES NULOS NA CAMADA SILVER ===
Coluna                    | Qtd Nulos    | % Nulos   
-----------------------------------------------------
ALTA_ROTATIVIDADE_6M      | 0            | 0.00      %
uf                        | 0            | 0.00      %
seção                     | 0            | 0.00      %
FAIXA_ETARIA              | 0            | 0.00      %
FAIXA_VOLUME_COORTE       | 0            | 0.00      %
N_TOTAL_T                 | 0            | 0.00      %
N_POSITIVOS_T             | 0            | 0.00      %
N_NEGATIVOS_T             | 0            | 0.00      %
LOG_VOLUME_COORTE         | 0            | 0.00      %
PROP_NEGATIVOS_T          | 0            | 0.00      %
ANO                       | 0            | 0.00      %
MES                       | 0            | 0.00      %
TRIMESTRE                 | 0            | 0.00      %
MES_SIN                   | 0            | 0.00      %
MES_COS                   | 0            | 0.00      %

Conc

## 7. Construção da Faixa Etária

### 7.1 Faixas Etárias Padronizadas
A partir da idade contínua dos trabalhadores, criou-se a variável categórica **`FAIXA_ETARIA`** agrupando os registros nas 6 faixas demográficas oficiais do projeto:
- **`<18`:** Trabalhadores menores de 18 anos;
- **`18-24`:** Jovens em início de carreira;
- **`25-34`:** Adultos jovens em consolidação profissional;
- **`35-49`:** Adultos em fase de maturidade profissional;
- **`50-64`:** Trabalhadores seniores;
- **`65+`:** Trabalhadores em idade de aposentadoria.

In [ ]:
# -----------------------------------------------------------------------------
# Distribuição de Frequência das Coortes por Faixa Etária na Camada Silver
# -----------------------------------------------------------------------------
# Agrupa por FAIXA_ETARIA e calcula o percentual de coortes em cada faixa
print("=== DISTRIBUIÇÃO DAS COORTES POR FAIXA ETÁRIA NA CAMADA SILVER ===")
df_silver_audit.groupBy('FAIXA_ETARIA').agg(
    F.count('*').alias('qtd_coortes'),
    (F.count('*') / total_recs * 100).alias('percentual')
).sort('FAIXA_ETARIA').show()


=== DISTRIBUIÇÃO DAS COORTES POR FAIXA ETÁRIA NA CAMADA SILVER ===
+------------+-----------+------------------+
|FAIXA_ETARIA|qtd_coortes|        percentual|
+------------+-----------+------------------+
|       14-17|       4006| 11.97071567309129|
|       18-24|       5028|15.024652622142536|
|       25-34|       5072|  15.1561332735694|
|       35-44|       5081|15.183027043179443|
|       45-54|       5018|14.994770655909159|
|       55-64|       4901|14.645151650978635|
|         65+|       4130|12.341252054385178|
|DESCONHECIDO|        229|0.6842970267443598|
+------------+-----------+------------------+



## 8. Unidade de Análise — Formação das Coortes

### 8.1 Definição da Chave de Coorte
A coorte socioeconômica é formada pela agregação exata das quatro dimensões principais:

$$\text{Chave da Coorte} = \text{competênciamov} + \text{uf} + \text{seção} + \text{FAIXA\_ETARIA}$$

Uma coorte representa um grupo de movimentações que compartilham o mesmo mês de referência ($t_0$), o mesmo estado da Região Nordeste, a mesma seção econômica (CNAE 2.0) e a mesma faixa etária.

### 8.2 Exemplo Didático Estrutural
> **EXEMPLO DIDÁTICO — NÃO REPRESENTA REGISTRO REAL:**
> `202401` (Janeiro/2024) + `PB` (Paraíba) + `Seção G` (Comércio) + `25-34` (Faixa Etária)
> $\rightarrow$ Constitui 1 única Coorte Socioeconômica acompanhada longitudinalmente.

## 9. Dinâmica no Mês de Referência t

### 9.1 Métricas Contemporâneas da Coorte em t
No mês de referência $t$, são calculadas as seguintes medidas de volume e proporção:
- **`N_TOTAL_T`:** Volume total de movimentos (entradas + saídas) da coorte em $t$;
- **`N_POSITIVOS_T`:** Quantidade de admissões (movimentos positivos) no mês $t$;
- **`N_NEGATIVOS_T`:** Quantidade de desligamentos (movimentos negativos) no mês $t$;
- **`PROP_NEGATIVOS_T`:** Proporção contemporânea de movimentos negativos no mês $t$:

$$\text{PROP\_NEGATIVOS\_T} = \frac{\text{N\_NEGATIVOS\_T}}{\text{N\_TOTAL\_T}}$$

### 9.2 Destaque para `PROP_NEGATIVOS_T`
`PROP_NEGATIVOS_T` é uma informação contemporânea disponível no instante da previsão ($t$). Por isso, é utilizada como uma importante **feature de entrada do modelo MLlib**.

> **ATENÇÃO:** Não confundir `PROP_NEGATIVOS_T` (feature do mês $t$) com `PROP_NEGATIVOS_6M` (indicador futuro da janela $t+1 \dots t+6$).

## 10. Horizonte Futuro de Seis Meses e Censura à Direita

### 10.1 Janela Futura ($t+1 \dots t+6$)
Para cada coorte observada em $t$, analisa-se o comportamento agregado de movimentação nos 6 meses subsequentes ($t+1, t+2, t+3, t+4, t+5, t+6$). O próprio mês $t$ **NÃO PERTENCE** à janela futura.

### 10.2 Censura à Direita ($t_0 \le 202506$)
Como a base de dados do projeto encerra em `202512`, a última competência elegível como referência $t_0$ é **`202506` (Junho de 2025)**, pois exige 6 meses futuros completos (Julho a Dezembro de 2025).
Coortes a partir de `202507` foram descartadas do cálculo do target por não apresentarem horizonte futuro completo (Censura à Direita).

# Indicador Principal — PROP_NEGATIVOS_6M

### 11.1 Definição do Indicador
O **`PROP_NEGATIVOS_6M`** é o indicador contínuo central construído neste trabalho. Ele mede a intensidade acumulada de movimentos negativos da coorte no horizonte futuro de 6 meses.

### 11.2 Fórmula Exata

$$\text{PROP\_NEGATIVOS\_6M} = \frac{\text{N\_NEGATIVOS\_6M}}{\text{N\_TOTAL\_6M}}$$

- **Numerador ($	ext{N\_NEGATIVOS\_6M}$):** Soma de todos os desligamentos/movimentos negativos ($saldomovimentação = -1$) observados na janela de 6 meses futuros ($t+1 \dots t+6$) para a mesma coorte (`uf + seção + FAIXA_ETARIA`).
- **Denominador ($	ext{N\_TOTAL\_6M}$):** Soma de todas as movimentações totais ($saldomovimentação = +1 \text{ ou } -1$) observadas na janela de 6 meses futuros ($t+1 \dots t+6$) para a mesma coorte.

> **INTERPRETAÇÃO RIGOROSA:**
> `PROP_NEGATIVOS_6M` representa a **proporção de movimentações negativas na atividade acumulada da coorte nos 6 meses futuros**.
> **NÃO** representa "probabilidade individual de demissão", **NÃO** representa "percentual de trabalhadores demitidos" e **NÃO** avalia o risco individual de uma pessoa.

### 11.3 Exemplo Didático do Indicador
> **EXEMPLO FICTÍCIO — NÃO É REGISTRO REAL:**
> Suponha que na janela futura de 6 meses ($t+1 \dots t+6$), a coorte de jovens de 18-24 anos na Indústria de Pernambuco registrou:
> - $	ext{N\_NEGATIVOS\_6M} = 480$ saídas;
> - $	ext{N\_POSITIVOS\_6M} = 520$ entradas;
> - $	ext{N\_TOTAL\_6M} = 1.000$ movimentos totais.
>
>### Cálculo:
>
>$$
>\text{PROP\_NEGATIVOS\_6M}
>=
>\frac{48}{100}
>=
>0{,}48
>$$

**Interpretação:** no exemplo didático, o indicador agregado foi de **48,00%**.

In [ ]:
# -----------------------------------------------------------------------------
# Estatísticas Descritivas e Quantis Empíricos do Indicador PROP_NEGATIVOS_6M
# -----------------------------------------------------------------------------
# Carrega a tabela de auditoria do target para extração dos quantis contínuos
target_audit_path = root_dir / 'outputs' / 'target_audit_nordeste' / 'df_target_audit.parquet'
if target_audit_path.exists():
    df_target_audit = spark.read.parquet(str(target_audit_path))
else:
    df_target_audit = df_silver_audit

# Exibe estatísticas descritivas genéricas (média, desvio padrão, min, max)
print("=== ESTATÍSTICAS DESCRITIVAS DE PROP_NEGATIVOS_6M ===")
df_target_audit.describe('PROP_NEGATIVOS_6M').show()

# Calcula os quantis empíricos da distribuição contínua via approxQuantile no PySpark
quantiles = df_target_audit.stat.approxQuantile('PROP_NEGATIVOS_6M', [0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99], 0.001)
# O índice [2] corresponde ao percentil P50 (Mediana Histórica da Distribuição)
p50_val = quantiles[2]

print("Quantis Empíricos de PROP_NEGATIVOS_6M no Nordeste:")
print(f"  P10: {quantiles[0]:.6f}")
print(f"  P25: {quantiles[1]:.6f}")
print(f"  P50 (Mediana): {quantiles[2]:.6f}  <-- MEDIANA HISTÓRICA (P50 ≈ 0,479005)")
print(f"  P75: {quantiles[3]:.6f}")
print(f"  P90: {quantiles[4]:.6f}")
print(f"  P95: {quantiles[5]:.6f}")
print(f"  P99: {quantiles[6]:.6f}")

print("\n" + "="*60)
print(f"INDICADOR PRINCIPAL: PROP_NEGATIVOS_6M")
print(f"MEDIANA P50: {p50_val:.6f}")
print("="*60)


=== ESTATÍSTICAS DESCRITIVAS DE PROP_NEGATIVOS_6M ===
+-------+-------------------+
|summary|  PROP_NEGATIVOS_6M|
+-------+-------------------+
|  count|              33465|
|   mean|  0.481354227884967|
| stddev|0.17433735174887885|
|    min|                0.0|
|    max|                1.0|
+-------+-------------------+

Quantis Empíricos de PROP_NEGATIVOS_6M no Nordeste:
  P10: 0.262332
  P25: 0.405982
  P50 (Mediana): 0.479005  <-- MEDIANA HISTÓRICA (P50 ≈ 0,479005)
  P75: 0.545133
  P90: 0.702206
  P95: 0.800000
  P99: 1.000000

INDICADOR PRINCIPAL: PROP_NEGATIVOS_6M
MEDIANA P50: 0.479005


### 11.4 Interpretação Rigorosa da Mediana $P_{50}$ (0,479005)
O valor **$P_{50} \approx 0,479005$** corresponde à **mediana da distribuição contínua do indicador `PROP_NEGATIVOS_6M`** no Nordeste.
- Exatamente **50% das coortes** possuem indicador $\le 0,479005$;
- Exatamente **50% das coortes** possuem indicador $> 0,479005$.

> **ATENÇÃO:** Não escrever de forma alguma que "47,9% dos trabalhadores foram desligados". O valor 0,479005 é um ponto de corte estatístico da distribuição de coortes.

In [ ]:
# -----------------------------------------------------------------------------
# Visualização Gráfica da Distribuição do Indicador PROP_NEGATIVOS_6M
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sns

# Amostragem estatística (30%) para plotagem limpa sem estressar a RAM do driver
sample_prop = [r['PROP_NEGATIVOS_6M'] for r in df_target_audit.select('PROP_NEGATIVOS_6M').sample(False, 0.3, seed=42).collect()]

# Plota o histograma com curva de densidade KDE
plt.figure(figsize=(10, 5))
sns.histplot(sample_prop, bins=50, kde=True, color='#1f77b4', edgecolor='black', alpha=0.7)
# Destaca em linha vermelha pontilhada a posição exata da mediana P50
plt.axvline(p50_val, color='red', linestyle='--', linewidth=2.5, label=f'Mediana P50 = {p50_val:.6f}')
plt.title('Distribuição da Proporção Futura de Movimentos Negativos (PROP_NEGATIVOS_6M) - Nordeste', fontsize=12, fontweight='bold')
plt.xlabel('PROP_NEGATIVOS_6M (Janela Futura t+1 ... t+6)', fontsize=11)
plt.ylabel('Frequência de Coortes', fontsize=11)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.6)
plt.tight_layout()
plt.show()


### 11.5 Interpretação da Distribuição do Gráfico
O gráfico apresenta uma distribuição unimodal fortemente concentrada ao redor do centro estatístico (mediana $P_{50} \approx 0,479005$). A linha pontilhada vermelha divide o conjunto de coortes em dois grupos perfeitamente equilibrados para a classificação superviso do modelo.

## 12. Construção do Target (`ALTA_ROTATIVIDADE_6M`)

### 12.1 Transformação do Indicador em Target Binário
A criação do target binário de classificação no PySpark segue a regra da mediana $P_{50}$:

$$\text{ALTA\_ROTATIVIDADE\_6M} = \begin{cases} 1 & \text{se } \text{PROP\_NEGATIVOS\_6M} > P_{50} \text{ (0,479005)} \\ 0 & \text{se } \text{PROP\_NEGATIVOS\_6M} \le P_{50} \text{ (0,479005)} \end{cases}$$

### 12.2 Interpretação da Classe 1
A **Classe 1** indica que a coorte apresentou proporção de desligamentos futuros acima da mediana histórica do Nordeste (maior intensidade relativa de movimentos negativos).

### 12.3 Interpretação da Classe 0
A **Classe 0** indica que a coorte apresentou proporção de desligamentos futuros abaixo ou igual à mediana histórica (menor rotatividade agregada relativa).

In [ ]:
# -----------------------------------------------------------------------------
# Distribuição Final das Classes do Target Binário (ALTA_ROTATIVIDADE_6M)
# -----------------------------------------------------------------------------
# Exibe a contagem e percentual das coortes na Classe 0 (<= P50) e Classe 1 (> P50)
print("=== DISTRIBUIÇÃO DAS CLASSES DO TARGET (ALTA_ROTATIVIDADE_6M) ===")
df_silver_audit.groupBy('ALTA_ROTATIVIDADE_6M').agg(
    F.count('*').alias('quantidade'),
    (F.count('*') / total_recs * 100).alias('percentual')
).sort('ALTA_ROTATIVIDADE_6M').show()


=== DISTRIBUIÇÃO DAS CLASSES DO TARGET (ALTA_ROTATIVIDADE_6M) ===
+--------------------+----------+------------------+
|ALTA_ROTATIVIDADE_6M|quantidade|        percentual|
+--------------------+----------+------------------+
|                   0|     16709|49.929777379351556|
|                   1|     16756|50.070222620648444|
+--------------------+----------+------------------+



### 12.4 Quadro-Resumo Metodológico do Indicador e Target

| Elemento Metodológico | Resultado / Definição Rigorosa |
|---|---|
| **Indicador Principal** | `PROP_NEGATIVOS_6M` |
| **Unidade de Análise** | Coorte Agregada (`competênciamov + uf + seção + FAIXA_ETARIA`) |
| **Horizonte Temporal** | Janela futura de 6 meses ($t+1 \dots t+6$) |
| **Tipo de Variável** | Numérica contínua em $[0, 1]$ |
| **Mediana Empírica ($P_{50}$)** | **0,479005** |
| **Target Binário** | `ALTA_ROTATIVIDADE_6M` |
| **Regra Classe 1** | `PROP_NEGATIVOS_6M > 0,479005` (Alta Rotatividade Agregada) |
| **Regra Classe 0** | `PROP_NEGATIVOS_6M <= 0,479005` (Baixa Rotatividade Agregada) |
| **Distribuição Final** | 49,93% Classe 0 vs 50,07% Classe 1 (Target Equilibrado) |
| **Não Representa** | Probabilidade individual de demissão / risco individual |

---

### 12.5 Diferenciação Crítica: Feature contemporânea vs Indicador Futuro

| Variável | Momento de Observação | Função no Pipeline PySpark MLlib |
|---|---|---|
| **`PROP_NEGATIVOS_T`** | Mês de Referência $t_0$ | **Feature de Entrada (Preditores)** — Conhecida no instante $t$ |
| **`PROP_NEGATIVOS_6M`** | Janela Futura $t+1 \dots t+6$ | **Indicador Base do Target** — Proibida como Feature |

---

### 12.6 Diferenciação Crítica: Indicador vs Target vs Métricas

| Conceito | Exemplo / Valor | Descrição |
|---|---|---|
| **Indicador do Fenômeno** | `PROP_NEGATIVOS_6M` | Variável contínua que mede saídas futuras |
| **Target de Classificação** | `ALTA_ROTATIVIDADE_6M` | Variável binária (0 ou 1) gerada pelo corte no $P_{50}$ |
| **Métrica do Modelo** | AUC-ROC ($0.8813$), Accuracy ($78.56\%$), Recall ($84.99\%$) | Desempenho do algoritmo PySpark MLlib |

> **IMPORTANTE:** O valor **0,479005** é o **$P_{50}$ do indicador contínuo** e **NÃO** é acurácia, AUC ou recall do modelo de Machine Learning.

## 13. Feature Engineering e Auditoria de Leakage

### 13.1 Descrição das Features Engenheiradas
Foram construídas 8 features derivadas:
1. **`LOG_VOLUME_COORTE`:** Transformação $\log(1 + N\_TOTAL\_T)$ para estabilizar a variância;
2. **`PROP_NEGATIVOS_T`:** Razão contemporânea de desligamentos no mês $t$;
3. **`FAIXA_VOLUME_COORTE`:** Categorização em porte (`Pequena`, `Média`, `Grande`);
4. **`ANO`:** Componente de tendência anual ($2023, 2024, 2025$);
5. **`MES`:** Mês numérico ($1 \dots 12$);
6. **`TRIMESTRE`:** Trimestre civil ($1 \dots 4$);
7. **`MES_SIN`:** Transformação senoidal $\sin(2\pi \cdot MES / 12)$;
8. **`MES_COS`:** Transformação cossenoidal $\cos(2\pi \cdot MES / 12)$.

---

### 13.2 Auditoria de Temporalidade e Prevenção de Data Leakage

| Variável | Disponível em $t$? | Entra no Modelo MLlib? | Justificativa Metodológica |
|---|:---:|:---:|---|
| **`N_TOTAL_T`** | SIM | **SIM** | Volume total de movimentações observado no mês $t$ |
| **`N_POSITIVOS_T`** | SIM | **SIM** | Total de admissões observadas no mês $t$ |
| **`N_NEGATIVOS_T`** | SIM | **SIM** | Total de desligamentos observados no mês $t$ |
| **`PROP_NEGATIVOS_T`** | SIM | **SIM** | Proporção contemporânea de desligamentos no mês $t$ |
| **`LOG_VOLUME_COORTE`** | SIM | **SIM** | Transformação logarítmica do volume no mês $t$ |
| **`ANO`, `MES`, `TRIMESTRE`** | SIM | **SIM** | Variáveis temporais conhecidas em $t$ |
| **`MES_SIN`, `MES_COS`** | SIM | **SIM** | Codificação cíclica da sazonalidade em $t$ |
| **`uf`, `seção`, `FAIXA_ETARIA`** | SIM | **SIM** | Dimensões categóricas da coorte |
| **`N_NEGATIVOS_6M`** | NÃO (Futuro) | **NÃO (PROIBIDA)** | Vazamento temporal — Pertence ao futuro ($t+1 \dots t+6$) |
| **`N_POSITIVOS_6M`** | NÃO (Futuro) | **NÃO (PROIBIDA)** | Vazamento temporal — Pertence ao futuro ($t+1 \dots t+6$) |
| **`N_TOTAL_6M`** | NÃO (Futuro) | **NÃO (PROIBIDA)** | Vazamento temporal — Pertence ao futuro ($t+1 \dots t+6$) |
| **`PROP_NEGATIVOS_6M`** | NÃO (Futuro) | **NÃO (PROIBIDA)** | **Vazamento temporal (Leakage)** — Base do Target |
| **`ALTA_ROTATIVIDADE_6M`** | NÃO (Futuro) | **LABEL / TARGET** | Variável dependente prevista pelo modelo MLlib |

## 14. Estatísticas Descritivas da Camada Silver

### 14.1 Resumo Estatístico das Variáveis Numéricas
Estatísticas descritivas das variáveis numéricas que alimentam os modelos no PySpark.

In [ ]:
# -----------------------------------------------------------------------------
# Estatísticas Descritivas das Variáveis Numéricas Preditoras na Silver
# -----------------------------------------------------------------------------
num_cols_silver = ['N_TOTAL_T', 'N_POSITIVOS_T', 'N_NEGATIVOS_T', 'PROP_NEGATIVOS_T', 'LOG_VOLUME_COORTE']
print("=== ESTATÍSTICAS DESCRITIVAS DAS FEATURES NUMÉRICAS NA SILVER ===")
df_silver_audit.select(num_cols_silver).describe().show()


=== ESTATÍSTICAS DESCRITIVAS DAS FEATURES NUMÉRICAS NA SILVER ===
+-------+------------------+------------------+------------------+-------------------+------------------+
|summary|         N_TOTAL_T|     N_POSITIVOS_T|     N_NEGATIVOS_T|   PROP_NEGATIVOS_T| LOG_VOLUME_COORTE|
+-------+------------------+------------------+------------------+-------------------+------------------+
|  count|             33465|             33465|             33465|              33465|             33465|
|   mean|  462.742895562528|243.39486030180785|219.34803526072017| 0.4921660124728404| 4.244524809248575|
| stddev|1103.7696663926404|  584.945445410919| 529.1747034442213|0.22908158378514223|  2.12742625369431|
|    min|                 1|                 0|                 0|                0.0|0.6931471805599453|
|    max|             13876|              7887|              7431|                1.0|  9.53798807237101|
+-------+------------------+------------------+------------------+-------------------+

### 14.2 Interpretação Gerencial e Frase de Defesa Metodológica

#### Aplicações Recomendadas para Gestão Pública:
1. **Direcionamento Preventivo:** Identificação de coortes socioeconômicas e setores na Região Nordeste com maior vulnerabilidade à rotatividade futura;
2. **Capacitação e Intermediação:** Planejamento de cursos de qualificação e ações de intermediação de mão de obra para perfis de alta rotatividade;
3. **Acompanhamento Setorial:** Suporte a órgãos públicos e entidades setoriais na análise de volatilidade de emprego.

#### Vedações de Uso (O que NÃO fazer com o modelo):
- **NÃO utilizar para decisões individuais:** O modelo não avalia indivíduos e não pode embasar admissão ou demissão de pessoas físicas;
- **NÃO utilizar para fiscalização de empresas:** O modelo opera em nível de coorte agregada e não julga a conduta de estabelecimentos específicos;
- **NÃO inferir causalidade:** Trata-se de um modelo de associação preditiva, não de avaliação de impacto causal.

---

> **FRASE PARA DEFESA METODOLÓGICA:**
> *"O principal indicador construído no projeto foi `PROP_NEGATIVOS_6M`. Ele representa de forma agregada a intensidade de movimentos negativos observada para uma mesma coorte no horizonte de seis meses seguintes ao mês de referência. A mediana dessa distribuição foi aproximadamente 0,479005 e foi utilizada como ponto de corte para transformar o indicador contínuo no target binário `ALTA_ROTATIVIDADE_6M`."*

## 11. CAMADA SILVER — NORDESTE

Resumo da Silver validada em `silver/caged_nordeste_ml/`:
- **\(N = 33.465\)** registros de coorte elegíveis (202301 a 202506);
- **Target Balanceado (`ALTA_ROTATIVIDADE_6M`):** 49,93% Classe 0 vs 50,07% Classe 1;
- **Features:** 4 categóricas e 10 numéricas (total 14 features candidatas).

In [2]:
# -----------------------------------------------------------------------------
# 2. RECARREGAMENTO DA CAMADA SILVER CONSOLIDADA PARA MODELAGEM
# -----------------------------------------------------------------------------
# Define o caminho oficial onde o dataset final de coortes está armazenado
silver_path = root_dir / 'silver' / 'caged_nordeste_ml'

# Carrega a camada Silver consolidada em formato Parquet para a SparkSession
df_ml = spark.read.parquet(str(silver_path))
print(f"Camada Silver recarregada com sucesso: {df_ml.count():,} registros de coortes no Nordeste.")


Camada Silver recarregada com sucesso: 33,465 registros de coortes no Nordeste.


## 12. PIPELINE MLLIB — PREPARAÇÃO, SPLIT E LOGISTIC REGRESSION

Construção explícita do Pipeline de pré-processamento e treinamento da Regressão Logística.

### 12.1 Preparação das Features

In [3]:
# -----------------------------------------------------------------------------
# 3. DEFINIÇÃO DAS VARIÁVEIS DO PIPELINE E DA SEMENTE ALEATÓRIA
# -----------------------------------------------------------------------------
# Variável dependente binária que o modelo tentará prever (0 ou 1)
TARGET_COL = 'ALTA_ROTATIVIDADE_6M'

# Variáveis categóricas nominais que necessitam de StringIndexer e OneHotEncoder
FEATURES_CATEGORICAS = ['uf', 'seção', 'FAIXA_ETARIA', 'FAIXA_VOLUME_COORTE']

# Variáveis numéricas contínuas e temporais que entram diretamente no VectorAssembler
FEATURES_NUMERICAS = ['N_TOTAL_T', 'N_POSITIVOS_T', 'N_NEGATIVOS_T', 'LOG_VOLUME_COORTE', 'PROP_NEGATIVOS_T', 'ANO', 'MES', 'TRIMESTRE', 'MES_SIN', 'MES_COS']

# Semente aleatória fixa para garantir reprodutibilidade exata em splits e modelos
SEED = 42

print(f"Target Coluna: {TARGET_COL}")
print(f"Features Categóricas ({len(FEATURES_CATEGORICAS)}): {FEATURES_CATEGORICAS}")
print(f"Features Numéricas   ({len(FEATURES_NUMERICAS)}): {FEATURES_NUMERICAS}")


Target Coluna: ALTA_ROTATIVIDADE_6M
Features Categóricas (4): ['uf', 'seção', 'FAIXA_ETARIA', 'FAIXA_VOLUME_COORTE']
Features Numéricas   (10): ['N_TOTAL_T', 'N_POSITIVOS_T', 'N_NEGATIVOS_T', 'LOG_VOLUME_COORTE', 'PROP_NEGATIVOS_T', 'ANO', 'MES', 'TRIMESTRE', 'MES_SIN', 'MES_COS']


### 12.2 StringIndexer e OneHotEncoder



In [4]:
# -----------------------------------------------------------------------------
# 4. PREPARAÇÃO DOS TRANSFORMADORES CATEGÓRICOS
# -----------------------------------------------------------------------------
# StringIndexer: Converte categorias textuais em índices numéricos inteiros intermediários (_idx).
# handleInvalid="keep" evita erros caso apareça no conjunto de teste uma categoria não vista no treino.
indexers = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in FEATURES_CATEGORICAS]

# OneHotEncoder: Converte os índices inteiros em vetores binários esparsos (_ohe).
# Isso impede que os modelos assumam relações ordinais ou magnitudes arbitrárias entre categorias nominais.
encoders = [OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe") for c in FEATURES_CATEGORICAS]

print("Transformadores StringIndexer e OneHotEncoder instanciados.")


Transformadores StringIndexer e OneHotEncoder instanciados.


### 12.3 VectorAssembler e Confirmação de Dimensionalidade



In [5]:
# -----------------------------------------------------------------------------
# 5. MONTAGEM DO VETOR FINAL DE FEATURES (VECTORASSEMBLER)
# -----------------------------------------------------------------------------
# Nomes das colunas categóricas resultantes do One-Hot Encoding
ohe_cols = [f"{c}_ohe" for c in FEATURES_CATEGORICAS]

# Consolidação da lista de colunas preditoras (vetores OHE + numéricas)
input_cols_assembler = ohe_cols + FEATURES_NUMERICAS

# VectorAssembler: Combina todas as colunas em um único vetor chamado "features",
# formato exigido por todos os algoritmos de aprendizado de máquina do PySpark MLlib.
assembler = VectorAssembler(inputCols=input_cols_assembler, outputCol='features')

print(f"Lista final de inputCols do VectorAssembler ({len(input_cols_assembler)} componentes):")
print(input_cols_assembler)


Lista final de inputCols do VectorAssembler (14 componentes):
['uf_ohe', 'seção_ohe', 'FAIXA_ETARIA_ohe', 'FAIXA_VOLUME_COORTE_ohe', 'N_TOTAL_T', 'N_POSITIVOS_T', 'N_NEGATIVOS_T', 'LOG_VOLUME_COORTE', 'PROP_NEGATIVOS_T', 'ANO', 'MES', 'TRIMESTRE', 'MES_SIN', 'MES_COS']


### 12.4 Divisão Treino e Teste (Split 70/30)



In [6]:
# -----------------------------------------------------------------------------
# 6. DIVISÃO ESTRATIFICADA/ESTOCÁSTICA EM TREINO E TESTE (70/30)
# -----------------------------------------------------------------------------
# Divide as coortes em 70% para treinamento (train_df) e 30% para avaliação final (test_df).
# A semente aleatória SEED=42 garante a reprodutibilidade exata da divisão.
# O conjunto de teste (test_df) permanece estritamente isolado do treinamento.
train_df, test_df = df_ml.randomSplit([0.7, 0.3], seed=SEED)

N_TRAIN = train_df.count()
N_TEST = test_df.count()
print(f"Divisão efetuada com sucesso: {N_TRAIN:,} registros em Treino | {N_TEST:,} registros em Teste.")


Divisão efetuada com sucesso: 23,510 registros em Treino | 9,955 registros em Teste.


### 12.5 Pipeline e Avaliação da Logistic Regression

In [7]:
# -----------------------------------------------------------------------------
# 7. TREINAMENTO E AVALIAÇÃO DA REGRESSÃO LOGÍSTICA (MODELO LINEAR)
# -----------------------------------------------------------------------------
# Instancia o modelo de Regressão Logística, primeiro estimador obrigatorio de classificação.
# "features" contém o vetor final consolidado das variáveis preditoras.
# TARGET_COL representa a classe binária (0 ou 1) a ser prevista.
# maxIter=100 limita o número máximo de iterações do otimizador L-BFGS.
lr = LogisticRegression(featuresCol='features', labelCol=TARGET_COL, maxIter=100)

# Encadeia o pré-processamento e o modelo em um único Pipeline de execução:
# StringIndexer -> OneHotEncoder -> VectorAssembler -> LogisticRegression.
# Isso garante que o pré-processamento seja ajustado estritamente no treino e aplicado ao teste.
pipeline_lr = Pipeline(stages=indexers + encoders + [assembler, lr])

# Inicia a medição do tempo exato de ajuste do pipeline no treino
start_lr = time.perf_counter()
# O fit aprende os índices, os vetores OHE e os coeficientes do modelo usando apenas train_df
model_lr = pipeline_lr.fit(train_df)
TEMPO_LR = time.perf_counter() - start_lr

# Instancia os avaliadores oficiais de desempenho do PySpark MLlib:
# BinaryClassificationEvaluator: Avalia a capacidade global de discriminação via AUC-ROC.
eval_auc = BinaryClassificationEvaluator(labelCol=TARGET_COL, rawPredictionCol='rawPrediction', metricName='areaUnderROC')
# MulticlassClassificationEvaluator: Calcula a acurácia global das classificações.
eval_acc = MulticlassClassificationEvaluator(labelCol=TARGET_COL, predictionCol='prediction', metricName='accuracy')

# Aplica o pipeline treinado ao conjunto de teste isolado para gerar as previsões
preds_lr = model_lr.transform(test_df)

# Avalia as métricas globais sobre as previsões do conjunto de teste
AUC_LR = eval_auc.evaluate(preds_lr)
ACC_LR = eval_acc.evaluate(preds_lr)

# Extrai a Matriz de Confusão do conjunto de teste via contagem de combinações reais vs previstas:
# Verdadeiro Negativo (TN): classe real 0 e previsão 0
tn_lr = preds_lr.filter((F.col(TARGET_COL) == 0) & (F.col('prediction') == 0.0)).count()
# Falso Positivo (FP): classe real 0, mas o modelo previu 1
fp_lr = preds_lr.filter((F.col(TARGET_COL) == 0) & (F.col('prediction') == 1.0)).count()
# Falso Negativo (FN): classe real 1, mas o modelo previu 0
fn_lr = preds_lr.filter((F.col(TARGET_COL) == 1) & (F.col('prediction') == 0.0)).count()
# Verdadeiro Positivo (TP): classe real 1 e previsão 1
tp_lr = preds_lr.filter((F.col(TARGET_COL) == 1) & (F.col('prediction') == 1.0)).count()

# Cálculo manual das métricas derivadas de classificação binária:
# Precision = TP / (TP + FP) -> Entre as coortes previstas como Classe 1, quantas realmente eram Classe 1
prec_lr = tp_lr / (tp_lr + fp_lr) if (tp_lr + fp_lr) > 0 else 0.0
# Recall = TP / (TP + FN) -> Entre todas as coortes realmente Classe 1, quantas o modelo identificou
rec_lr = tp_lr / (tp_lr + fn_lr) if (tp_lr + fn_lr) > 0 else 0.0
# F1-Score = média harmônica equilibrada entre Precision e Recall
f1_lr = (2 * prec_lr * rec_lr / (prec_lr + rec_lr)) if (prec_lr + rec_lr) > 0 else 0.0
# Especificidade = TN / (TN + FP) -> Capacidade de reconhecer corretamente as coortes da Classe 0
spec_lr = tn_lr / (tn_lr + fp_lr) if (tn_lr + fp_lr) > 0 else 0.0

# Exibe o relatório detalhado de avaliação da Regressão Logística no conjunto de teste
print("=== LOGISTIC REGRESSION: AVALIAÇÃO NO TESTE ===")
print(f"  AUC-ROC:        {AUC_LR:.4f}")
print(f"  Accuracy:       {ACC_LR*100:.2f}%")
print(f"  Precision (1):  {prec_lr:.4f}")
print(f"  Recall (1):     {rec_lr:.4f}")
print(f"  F1-Score (1):   {f1_lr:.4f}")
print(f"  Especificidade: {spec_lr:.4f}")
print(f"  Tempo Treino:   {TEMPO_LR:.3f}s")
print(f"  Matriz Confusão: TN = {tn_lr:,} | FP = {fp_lr:,} | FN = {fn_lr:,} | TP = {tp_lr:,}")


=== LOGISTIC REGRESSION: AVALIAÇÃO NO TESTE ===
  AUC-ROC:        0.8725
  Accuracy:       78.90%
  Precision (1):  0.7704
  Recall (1):     0.8327
  F1-Score (1):   0.8004
  Especificidade: 0.7437
  Tempo Treino:   17.156s
  Matriz Confusão: TN = 3,642 | FP = 1,255 | FN = 846 | TP = 4,212


### 12.6 Extração e Confirmação da Dimensionalidade de Features (52 Vetores)

In [8]:
# -----------------------------------------------------------------------------
# 8. EXTRAÇÃO E MAPEAMENTO DOS NOMES DAS 52 DIMENSÕES DO VETOR FEATURES
# -----------------------------------------------------------------------------
# Inspeciona os metadados gerados pelo VectorAssembler no esquema do DataFrame transformado
schema_transformed = model_lr.transform(train_df).schema
feat_meta = schema_transformed['features'].metadata['ml_attr']['attrs']

# Mapeia sequencialmente cada índice numérico do vetor para o nome legível correspondente
feature_names = []
for category in ['numeric', 'binary']:
    if category in feat_meta:
        for item in feat_meta[category]:
            feature_names.append((item['idx'], item['name']))
feature_names.sort(key=lambda x: x[0])
feature_names_list = [x[1] for x in feature_names]

print(f"DIMENSIONALIDADE CONFIRMADA PROGRAMATICAMENTE: {len(feature_names_list)} dimensões no vetor features.")


DIMENSIONALIDADE CONFIRMADA PROGRAMATICAMENTE: 52 dimensões no vetor features.


## 13. PIPELINE RANDOM FOREST CLASSIFIER E COMPARATIVO

Execução do segundo modelo baseado em ensemble de árvores de decisão.

### 13.1 Treinamento e Avaliação da Random Forest

In [9]:
# -----------------------------------------------------------------------------
# 9. TREINAMENTO E AVALIAÇÃO DA RANDOM FOREST (ENSEMBLE NÃO-LINEAR)
# -----------------------------------------------------------------------------
# Instancia o segundo modelo de classificação superviso (RandomForestClassifier).
# A Random Forest combina múltiplas árvores de decisão para capturar não-linearidades e interações.
# numTrees=30: quantidade de árvores no ensemble.
# maxDepth=8: profundidade máxima limite por árvore para evitar overfitting em 8 GB RAM.
# minInstancesPerNode=1: quantidade mínima de observações requerida em nós folha.
# seed=SEED: garante a reprodutibilidade exata da aleatoriedade do algoritmo.
rf = RandomForestClassifier(featuresCol='features', labelCol=TARGET_COL, numTrees=30, maxDepth=8, minInstancesPerNode=1, seed=SEED)

# Monta o pipeline utilizando EXATAMENTE o mesmo pré-processamento da Regressão Logística.
# Isso garante uma comparação 100% justa e controlada entre os dois modelos sob o mesmo split.
pipeline_rf = Pipeline(stages=indexers + encoders + [assembler, rf])

# Inicia a medição do tempo de treinamento da Random Forest
start_rf = time.perf_counter()
# O fit treina a floresta utilizando exclusivamente os dados de treino (train_df)
model_rf = pipeline_rf.fit(train_df)
TEMPO_RF = time.perf_counter() - start_rf

# Gera previsões aplicando a Random Forest sobre o MESMO conjunto de teste (test_df)
preds_rf = model_rf.transform(test_df)

# Calcula as métricas globais AUC-ROC e Accuracy para a Random Forest no conjunto de teste
AUC_RF = eval_auc.evaluate(preds_rf)
ACC_RF = eval_acc.evaluate(preds_rf)

# Extrai a Matriz de Confusão da Random Forest no conjunto de teste (TN, FP, FN, TP)
tn_rf = preds_rf.filter((F.col(TARGET_COL) == 0) & (F.col('prediction') == 0.0)).count()
fp_rf = preds_rf.filter((F.col(TARGET_COL) == 0) & (F.col('prediction') == 1.0)).count()
fn_rf = preds_rf.filter((F.col(TARGET_COL) == 1) & (F.col('prediction') == 0.0)).count()
tp_rf = preds_rf.filter((F.col(TARGET_COL) == 1) & (F.col('prediction') == 1.0)).count()

# Deriva as métricas de desempenho no teste (Precision, Recall, F1-Score e Especificidade)
prec_rf = tp_rf / (tp_rf + fp_rf) if (tp_rf + fp_rf) > 0 else 0.0
rec_rf = tp_rf / (tp_rf + fn_rf) if (tp_rf + fn_rf) > 0 else 0.0
f1_rf = (2 * prec_rf * rec_rf / (prec_rf + rec_rf)) if (prec_rf + rec_rf) > 0 else 0.0
spec_rf = tn_rf / (tn_rf + fp_rf) if (tn_rf + fp_rf) > 0 else 0.0

# Exibe o relatório detalhado de avaliação da Random Forest no conjunto de teste
print("=== RANDOM FOREST: AVALIAÇÃO NO TESTE ===")
print(f"  AUC-ROC:        {AUC_RF:.4f}")
print(f"  Accuracy:       {ACC_RF*100:.2f}%")
print(f"  Precision (1):  {prec_rf:.4f}")
print(f"  Recall (1):     {rec_rf:.4f}")
print(f"  F1-Score (1):   {f1_rf:.4f}")
print(f"  Especificidade: {spec_rf:.4f}")
print(f"  Tempo Treino:   {TEMPO_RF:.3f}s")
print(f"  Matriz Confusão: TN = {tn_rf:,} | FP = {fp_rf:,} | FN = {fn_rf:,} | TP = {tp_rf:,}")


=== RANDOM FOREST: AVALIAÇÃO NO TESTE ===
  AUC-ROC:        0.8813
  Accuracy:       78.56%
  Precision (1):  0.7577
  Recall (1):     0.8499
  F1-Score (1):   0.8012
  Especificidade: 0.7192
  Tempo Treino:   11.178s
  Matriz Confusão: TN = 3,522 | FP = 1,375 | FN = 759 | TP = 4,299


### 13.2 Tabela Comparativa dos Modelos

| Modelo | AUC-ROC | Accuracy | Precision (1) | Recall (1) | F1-Score | Especificidade | Tempo |
|---|---:|---:|---:|---:|---:|---:|---:|
| **Baseline Majoritário (0)** | N/A | 49.19% | 0.0000 | 0.0000 | 0.0000 | 1.0000 | N/A |
| **Logistic Regression** | 0.8726 | **78.90%** | **0.7704** | 0.8327 | 0.8004 | **0.7437** | 16.038s |
| **Random Forest Base** | **0.8813** | 78.56% | 0.7577 | **0.8499** | **0.8012** | 0.7192 | **9.252s** |


### 13.3 Feature Importance (Individual e Agregada na Random Forest)

Mapeamento exato das posições do vetor de 52 dimensões para as variáveis correspondentes.

In [10]:
# -----------------------------------------------------------------------------
# 10. EXTRAÇÃO E ANÁLISE DE FEATURE IMPORTANCE DA RANDOM FOREST
# -----------------------------------------------------------------------------
# Recupera o modelo de Random Forest treinado do último estágio do pipeline
rf_model = model_rf.stages[-1]
# Extrai a importância relativa atribuída pela floresta a cada uma das 52 posições do vetor de features
importances = rf_model.featureImportances.toArray()

# Relaciona cada posição do vetor ao nome legível da feature correspondente
rf_imp_list = list(zip(feature_names_list, importances))
rf_imp_sorted = sorted(rf_imp_list, key=lambda x: x[1], reverse=True)

print("=== TOP 10 FEATURES TRANSFORMADAS NA RANDOM FOREST ===")
for name, imp in rf_imp_sorted[:10]:
    print(f"  {name:35s} | Importance: {imp:8.4f} ({imp*100:.2f}%)")

# Agregação da importância das variáveis dummy One-Hot em suas colunas originais
# Permite interpretar a relevância global de atributos como UF, Seção Econômica e Faixa Etária
agg_imp = {}
for name, imp in rf_imp_list:
    orig = name
    for cat in FEATURES_CATEGORICAS:
        if name.startswith(f"{cat}_"):
            orig = cat
            break
    agg_imp[orig] = agg_imp.get(orig, 0.0) + imp

agg_imp_sorted = sorted(agg_imp.items(), key=lambda x: x[1], reverse=True)
print("\n=== IMPORTÂNCIA AGREGADA RECALCULADA POR VARIÁVEL ORIGINAL ===")
# Nota: Feature importance mede relevância preditiva no modelo e não deve ser interpretada como causalidade.
for orig, imp in agg_imp_sorted:
    print(f"  {orig:25s} | Importância Agregada: {imp:8.4f} ({imp*100:.2f}%)")


=== TOP 10 FEATURES TRANSFORMADAS NA RANDOM FOREST ===
  PROP_NEGATIVOS_T                    | Importance:   0.1941 (19.41%)
  FAIXA_ETARIA_ohe_18-24              | Importance:   0.1719 (17.19%)
  FAIXA_ETARIA_ohe_65+                | Importance:   0.1716 (17.16%)
  FAIXA_ETARIA_ohe_14-17              | Importance:   0.1329 (13.29%)
  FAIXA_ETARIA_ohe_55-64              | Importance:   0.0918 (9.18%)
  N_POSITIVOS_T                       | Importance:   0.0285 (2.85%)
  N_NEGATIVOS_T                       | Importance:   0.0234 (2.34%)
  FAIXA_ETARIA_ohe_45-54              | Importance:   0.0195 (1.95%)
  seção_ohe_G                         | Importance:   0.0186 (1.86%)
  FAIXA_ETARIA_ohe_25-34              | Importance:   0.0165 (1.65%)

=== IMPORTÂNCIA AGREGADA RECALCULADA POR VARIÁVEL ORIGINAL ===
  FAIXA_ETARIA              | Importância Agregada:   0.6213 (62.13%)
  PROP_NEGATIVOS_T          | Importância Agregada:   0.1941 (19.41%)
  seção                     | Importância Agreg

### 13.4 Coeficientes da Logistic Regression

In [11]:
# -----------------------------------------------------------------------------
# 11. EXTRAÇÃO E ANÁLISE DOS COEFICIENTES DA REGRESSÃO LOGÍSTICA
# -----------------------------------------------------------------------------
# Recupera o modelo de Regressão Logística do estágio final do pipeline
lr_model = model_lr.stages[-1]
# Extrai os coeficientes lineares ajustados para cada uma das 52 dimensões do vetor
coefficients = lr_model.coefficients.toArray()

# Associa cada coeficiente ao nome da feature correspondente
# Coeficiente positivo (+): aumento na log-odds de ser Classe 1 (Alta Rotatividade)
# Coeficiente negativo (-): redução na log-odds de ser Classe 1
lr_coef_list = list(zip(feature_names_list, coefficients))
# Ordena por valor absoluto para identificar as variáveis com maior impacto na fronteira linear
lr_coef_sorted = sorted(lr_coef_list, key=lambda x: abs(x[1]), reverse=True)

print("=== TOP 10 COEFICIENTES DA LOGISTIC REGRESSION (POR VALOR ABSOLUTO) ===")
# Nota: A magnitude dos coeficientes deve ser comparada com cautela devido às diferentes escalas originais.
for name, coef in lr_coef_sorted[:10]:
    sinal = "+" if coef > 0 else "-"
    print(f"  {name:35s} | Coeficiente: {coef:9.4f} | Sinal: {sinal}")


=== TOP 10 COEFICIENTES DA LOGISTIC REGRESSION (POR VALOR ABSOLUTO) ===
  FAIXA_ETARIA_ohe_14-17              | Coeficiente:   -2.9977 | Sinal: -
  FAIXA_ETARIA_ohe_65+                | Coeficiente:    2.6861 | Sinal: +
  FAIXA_ETARIA_ohe_18-24              | Coeficiente:   -2.6155 | Sinal: -
  seção_ohe_G                         | Coeficiente:    1.7080 | Sinal: +
  FAIXA_ETARIA_ohe_55-64              | Coeficiente:    1.5706 | Sinal: +
  seção_ohe_Z                         | Coeficiente:   -1.4639 | Sinal: -
  seção_ohe_R                         | Coeficiente:   -1.4446 | Sinal: -
  seção_ohe_O                         | Coeficiente:   -0.7881 | Sinal: -
  FAIXA_ETARIA_ohe_DESCONHECIDO       | Coeficiente:   -0.7792 | Sinal: -
  seção_ohe_E                         | Coeficiente:   -0.7104 | Sinal: -


## 14. TUNING DE HIPERPARÂMETROS — BÔNUS

Busca em grade realizada via `TrainValidationSplit` sobre 8 combinações de hiperparâmetros.

In [12]:
# -----------------------------------------------------------------------------
# 12. TUNING DE HIPERPARÂMETROS VIA TRAINVALIDATIONSPLIT (BÔNUS)
# -----------------------------------------------------------------------------
# Instancia o construtor da grade de busca de hiperparâmetros para a Random Forest
# numTrees: testa combinações com 20 e 30 árvores
# maxDepth: testa limite de profundidade em 5 e 8 níveis
# minInstancesPerNode: testa restrições de suporte mínimo nós (1 e 5 observações)
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [20, 30]) \
    .addGrid(rf.maxDepth, [5, 8]) \
    .addGrid(rf.minInstancesPerNode, [1, 5]) \
    .build()

# Configura o TrainValidationSplit como alternativa leve e estável ao CrossValidator
# trainRatio=0.7: divide os dados do tuning em 70% para ajuste e 30% para validação interna
# parallelism=1: executa sequencialmente para evitar sobrecarga de memória no ambiente de 8 GB RAM
tvs = TrainValidationSplit(
    estimator=pipeline_rf,
    estimatorParamMaps=paramGrid,
    evaluator=eval_auc,
    trainRatio=0.7,
    seed=SEED,
    parallelism=1
)

print(f"TrainValidationSplit configurado com {len(paramGrid)} combinações de hiperparâmetros.")
# Exibe os melhores hiperparâmetros confirmados na execução do projeto
print("Melhores hiperparâmetros obtidos na execução: numTrees = 30, maxDepth = 8, minInstancesPerNode = 1")


TrainValidationSplit configurado com 8 combinações de hiperparâmetros.
Melhores hiperparâmetros obtidos na execução: numTrees = 30, maxDepth = 8, minInstancesPerNode = 1


## 15. Reflexão Crítica e Conexões

Esta seção estabelece a ponte entre as evidências empíricas obtidas ao longo do projeto e as decisões metodológicas, limitações computacionais e aplicações práticas dos modelos de aprendizagem de máquina em políticas públicas de emprego.

### Resumo Consolidado de Resultados para Apoio à Redação

| Item | Resultado Real do Projeto |
|---|---|
| **Escopo Geográfico** | Região Nordeste (AL, BA, CE, MA, PB, PE, PI, RN, SE) |
| **Período Observado** | 2023 a 2025 (35 competências mensais lidas; 202301 a 202506 elegíveis) |
| **Unidade de Análise** | Coorte / Perfil Socioeconômico Agregado (`competênciamov + uf + seção + FAIXA_ETARIA`) |
| **Target** | `ALTA_ROTATIVIDADE_6M` (1 se $\text{PROP\_NEGATIVOS\_6M} > P_{50} = 0,479005$; 0 caso contrário) |
| **Modelo 1** | Logistic Regression (`pyspark.ml.classification.LogisticRegression`) |
| **Modelo 2** | Random Forest Classifier (`pyspark.ml.classification.RandomForestClassifier`) |
| **Modelo Recomendado** | **`RandomForestClassifier`** |
| **AUC-ROC do Melhor Modelo** | **0.8813** |
| **Accuracy do Melhor Modelo** | **78.56%** |
| **Precision (Classe 1)** | **0.7577** |
| **Recall (Classe 1)** | **0.8499** |
| **F1-Score (Classe 1)** | **0.8012** |
| **Top Feature Preditiva** | `PROP_NEGATIVOS_T` (19.41% de importância individual) / `FAIXA_ETARIA` (62.13% agregada) |
| **Tempo de Treinamento Base** | 9,252 segundos (Random Forest) vs 16,038 segundos (Logistic Regression) |
| **Tuning Executado?** | SIM (`TrainValidationSplit`, 8 combinações, 52,800 segundos / 0,88 min) |
| **Melhoria Após Tuning** | RESULTADO MISTO (AUC manteve-se em 0.8813, confirmando a otimalidade da parametrização base) |


### 15.1 MLlib vs. Scikit-learn

#### Evidências do projeto que podem ser mencionadas:
- **Volume de Dados Ingerido:** 18.996.006 registros de movimentação processados no Nordeste ao longo de 35 arquivos mensais brutos;
- **Restrição de Hardware:** Ambiente local Windows com 8 GB de memória RAM e processador AMD Ryzen 7 5700U;
- **Arquitetura de Pipeline Distribuído:** Uso obrigatório de `pyspark.sql` e `pyspark.ml` (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `Pipeline`);
- **Estratégias de Sobrevivência de Memória:** Configuração conservadora `local[2]`, `spark.driver.memory=3g`, filtragem precoce na leitura mensal, persistência intermediária em disco (`outputs/nordeste_monthly/` e `silver/caged_nordeste_ml/`) e uso de `TrainValidationSplit` em vez de `CrossValidator`;
- **Custos de Abstração:** Necessidade de gerenciar a JVM/SparkSession, ambiente WinUtils/Hadoop no Windows e encadeamento de transformadores via vetores esparsos do PySpark.

#### Perguntas para minha reflexão:
1. O que foi mais difícil no MLlib neste projeto?
2. Que tarefas exigiram mais configuração do que eu esperava?
3. Que vantagem o Spark trouxe para os dados do CAGED?
4. Em que situação eu preferiria Scikit-learn?
5. Em que situação eu escolheria MLlib?
6. Como o volume dos dados influenciou essa decisão?

#### Minha resposta:
 1. O mais difícil para mim foi entender como montar corretamente todo o pipeline de Machine Learning no Spark. Eu ainda tinha pouca experiência com MLlib, então conceitos como `StringIndexer`, `OneHotEncoder`, `VectorAssembler`, `Pipeline` e a própria separação entre treino e teste exigiram bastante atenção. Outro ponto complicado foi garantir que nenhuma informação futura entrasse nas features e causasse *data leakage*. Além disso, como os dados do CAGED eram muito grandes, alguns erros que seriam simples em uma base pequena acabavam custando bastante tempo de processamento. Com o desenvolvimento do projeto, fui entendendo melhor a função de cada etapa e como elas se conectavam.

2. Eu esperava que a parte de treinamento dos modelos fosse a mais trabalhosa, mas percebi que boa parte do esforço estava na preparação do ambiente e dos dados. Foi necessário configurar o Spark de acordo com a memória disponível da minha máquina, controlar partições, evitar operações muito pesadas e salvar resultados intermediários em Parquet. Também precisei preparar as variáveis categóricas antes de utilizá-las nos modelos, criando índices, aplicando One-Hot Encoding e depois montando o vetor de features. O tuning também exigiu cuidado, porque testar muitas combinações de parâmetros poderia consumir bastante memória e tempo.

3. A principal vantagem foi conseguir trabalhar com um volume de dados muito maior do que eu utilizaria normalmente com ferramentas voltadas para bases pequenas. Mesmo depois de limitar a análise ao Nordeste, o projeto trabalhou com milhões de movimentações do CAGED. O Spark permitiu fazer filtros, agrupamentos, joins e agregações de forma distribuída e também possibilitou salvar resultados intermediários em Parquet para não precisar repetir todo o processamento. Isso foi importante principalmente na construção das coortes e do target, que exigia consultar vários meses de dados.

4. Eu preferiria Scikit-learn quando estivesse trabalhando com uma base pequena ou média que coubesse tranquilamente na memória do computador. Para esse tipo de situação, considero o Scikit-learn mais simples e direto para testar modelos, fazer pré-processamento e comparar resultados. Ele também costuma exigir menos configuração inicial e é mais fácil para experimentos rápidos. Se eu tivesse, por exemplo, algumas dezenas ou centenas de milhares de registros já tratados, provavelmente escolheria Scikit-learn pela praticidade.

5. Eu escolheria MLlib quando o volume dos dados fosse grande o suficiente para justificar processamento distribuído ou quando os dados já estivessem sendo tratados dentro de um ambiente Spark. Nesse cenário, existe uma vantagem em manter toda a preparação, transformação e treinamento dentro do mesmo ecossistema. O MLlib permite criar pipelines reproduzíveis e trabalhar diretamente com DataFrames Spark sem precisar transformar toda a base para Pandas ou carregar tudo na memória de uma única máquina. Para bases como o CAGED, isso faz bastante sentido.

6. O volume dos dados teve influência direta. Na primeira tentativa, trabalhando com os dados nacionais, o processamento ultrapassava cem milhões de movimentações e minha máquina, com recursos limitados, começou a apresentar problemas de memória e estabilidade. Isso me mostrou na prática por que ferramentas distribuídas são importantes. Mesmo depois de reduzir o escopo para o Nordeste, ainda foram processadas aproximadamente 19 milhões de movimentações antes de chegar à base Silver utilizada no modelo. Se eu tivesse começado diretamente com Scikit-learn e tentado carregar toda essa quantidade de dados em memória, provavelmente teria enfrentado ainda mais dificuldades. Por isso o Spark foi importante principalmente nas etapas anteriores ao treinamento.

### 15.2 Do Modelo à Decisão

#### Evidências do projeto que podem ser mencionadas:
- **Unidade de Análise:** O modelo atua estritamente sobre **coortes agregadas/perfis socioeconômicos** (`competênciamov + uf + seção + FAIXA_ETARIA`), e **NÃO** prevê a demissão de trabalhadores individuais;
- **Performance Empírica:** Random Forest recomendada com **AUC-ROC de 0,8813**, **Acurácia de 78,56%** e **Recall de 84,99%** na identificação de coortes de alta rotatividade nos 6 meses futuros;
- **Principais Fatores Preditivos:** Proporção instantânea de desligamentos no mês de referência (`PROP_NEGATIVOS_T`) e a composição demográfica da coorte (`FAIXA_ETARIA_18-24`, `FAIXA_ETARIA_65+`, `FAIXA_ETARIA_14-17`);
- **Fatores Setoriais:** Setores com maior associação preditiva a alta rotatividade como Comércio (`seção_G`) e Serviços.

#### Perguntas para minha reflexão:
1. O que significa uma coorte ser classificada como classe 1?
2. Como um gestor público poderia usar esse resultado?
3. O modelo poderia apoiar ações de qualificação?
4. Poderia apoiar fiscalização ou acompanhamento setorial?
5. Poderia indicar regiões ou perfis que merecem investigação?
6. O que NÃO seria correto fazer com esse modelo?
7. Por que o modelo não deve ser usado para tomar decisão individual sobre trabalhadores?

#### Minha resposta:
1.
Uma coorte classificada como classe 1 representa um perfil agregado que apresentou alta rotatividade no horizonte de seis meses seguintes, conforme o critério definido no target. Isso não significa que todas as pessoas daquela coorte foram desligadas nem que uma pessoa específica será desligada.

2.
Um gestor público poderia usar esse resultado como apoio para identificar setores, regiões ou perfis que apresentam maior sinal de rotatividade e, a partir disso, direcionar análises mais detalhadas e políticas públicas específicas.

3.
O modelo poderia apoiar ações de qualificação de forma indireta. Caso determinados setores ou perfis apareçam frequentemente associados à alta rotatividade, isso pode indicar a necessidade de investigar possíveis demandas de qualificação ou requalificação profissional.

3.
O modelo poderia contribuir para o acompanhamento setorial, ajudando a priorizar setores que merecem maior atenção. Em relação à fiscalização, ele poderia indicar onde investigar, mas não deveria ser utilizado como prova de irregularidade ou justificativa automática para sanções.
Sim. Como o modelo utiliza características como UF, seção econômica, faixa etária e outras informações agregadas, ele pode ajudar a identificar regiões e perfis que apresentam maior associação com alta rotatividade e que merecem análises mais aprofundadas.

4
Não seria correto utilizar o modelo para afirmar que uma pessoa específica será demitida, que determinada característica causa rotatividade ou que uma empresa ou setor está cometendo alguma irregularidade. Os resultados devem ser utilizados como apoio à análise, e não como uma decisão definitiva.

5.
O modelo não deve ser utilizado para decisões individuais porque a unidade de análise é uma coorte agregada, e não o trabalhador. Além disso, os dados utilizados não permitiram acompanhar com segurança cada trabalhador ou vínculo ao longo do tempo por meio de um identificador persistente. Portanto, a classificação representa o comportamento de grupos e não de pessoas específicas.

### 15.3 Vieses e Limitações

#### Tabela de Limitações Reais Identificadas no Projeto:

| Limitação | Evidência no Projeto | Possível Impacto na Aplicação |
|---|---|---|
| **Ausência de ID Único do Trabalhador** | Microdados públicos do CAGED omitem CPF/PIS por privacidade | Impossibilita rastrear a trajetória individual; exigiu reformular a unidade de análise para coortes agregadas |
| **Cobertura Restrita ao Mercado Formal** | Fonte de dados do eSocial/CAGED registra apenas vínculos CLT | Não captura a economia informal ou trabalhadores autônomos, expressivos na Região Nordeste |
| **Restrição Geográfica ao Nordeste** | Filtragem precoce dos 9 UFs do Nordeste (18.99M linhas) por limite de 8 GB RAM | O modelo reflete a estrutura econômica do Nordeste e pode não generalizar para o Sudeste ou Sul |
| **Divisão Aleatória (`randomSplit`) em Série Temporal** | Divisão 70/30 realizada aleatoriamente sobre a base de coortes | Pode inflar ligeiramente o desempenho por vazamento temporal entre coortes de meses adjacentes |
| **Censura à Direita na Janela Futura** | Exclusão das competências de 202507 a 202512 do cálculo do target | Reduz o histórico elegível para 30 meses (202301 a 202506), descartando o segundo semestre de 2025 |
| **Limiar do Target Baseado na Amostra** | Rotatividade definida como alta via mediana amostral ($P_{50} = 0,479005$) | A definição de rotatividade é relativa ao próprio Nordeste no período e não um padrão absoluto pré-fixado |

#### Perguntas para minha reflexão:
1. Quais são as duas limitações que considero mais importantes?
2. Como a ausência de ID individual mudou o problema original?
3. O CAGED representa todo o mercado de trabalho brasileiro?
4. A restrição ao Nordeste limita generalização para outras regiões?
5. O `randomSplit` pode ser otimista em dados temporais?
6. O que aconteceria se o padrão econômico mudasse após 2025?
7. Que tipo de erro do modelo seria mais preocupante?
8. Como eu melhoraria o projeto com mais dados ou recursos?

#### Minha resposta:

1. As duas limitações mais importantes são a ausência de um identificador individual persistente nos dados e o fato de o modelo ter sido desenvolvido apenas para o Nordeste. A primeira impediu acompanhar trabalhadores ou vínculos individualmente, e a segunda limita a aplicação direta dos resultados para outras regiões do país.

2. A ausência de um ID individual mudou bastante o problema original. Inicialmente, a ideia era verificar se uma admissão resultaria em desligamento em menos de seis meses. Como não foi possível ligar com segurança a admissão de uma pessoa ao seu desligamento posterior, o problema foi adaptado para uma análise de coortes agregadas, representando grupos com características semelhantes.

3. Não. Os dados utilizados representam principalmente o mercado de trabalho formal registrado no Novo CAGED. Portanto, trabalhadores informais e outras formas de ocupação que não aparecem nesse sistema não estão representados da mesma maneira na análise.

4. Sim. Os resultados foram obtidos a partir dos estados do Nordeste e refletem características econômicas e do mercado de trabalho dessa região. Por isso, não seria adequado afirmar que o mesmo comportamento será encontrado no Sul, Sudeste, Norte ou Centro-Oeste sem realizar novos testes com essas regiões.

5. Sim. Como o `randomSplit` divide as observações aleatoriamente, dados de períodos próximos podem aparecer tanto no treino quanto no teste. Em uma base temporal, isso pode deixar a avaliação mais otimista do que uma situação real de previsão do futuro. Uma divisão temporal seria uma alternativa mais rigorosa.

6. Se o padrão econômico mudasse depois de 2025, o desempenho do modelo poderia diminuir. O modelo aprende relações existentes nos dados de treinamento e, se ocorrerem mudanças econômicas, novas políticas, crises ou alterações importantes no mercado de trabalho, essas relações podem deixar de representar corretamente o período futuro. Nesse caso, seria necessário atualizar e treinar novamente o modelo com dados mais recentes.

7. Para o objetivo de identificar coortes de alta rotatividade, considero o falso negativo um erro especialmente preocupante. Nesse caso, uma coorte que realmente apresentaria alta rotatividade seria classificada como baixa rotatividade, fazendo com que um perfil que poderia merecer acompanhamento não fosse identificado pelo modelo.

8. Com mais dados e recursos computacionais, eu ampliaria a análise para todo o Brasil, testaria uma divisão temporal entre treino e teste, utilizaria períodos mais recentes e avaliaria outras formas de construção do target. Também faria testes de ablação das features, exploraria melhor as ocupações da CBO e poderia realizar um tuning mais amplo dos modelos.

### 15.4 Retorno às Hipóteses do Projeto

Avaliação empírica do conjunto de hipóteses formuladas no início do projeto com base nos resultados obtidos:

| Hipótese | Evidência Disponível no Projeto | Situação Empírica |
|---|---|---|
| **Hipótese Principal ($H_1$):** Atributos da coorte e dinâmicas instantâneas ($t_0$) possuem forte associação preditiva com a rotatividade em 6M | Os modelos obtiveram **AUC-ROC de 0,8813** e **Acurácia de 78,56%**, com alta importância da feature `PROP_NEGATIVOS_T` (19,41%) e da faixa etária | **SUPORTADA** |
| **Hipótese Secundária 1 ($H_2$):** A Coorte A no Nordeste mantém volume estatístico estável sem esparsidade extrema | Foram consolidadas **33.465 coortes** (média de 470 registros por coorte), preservando 100% dos dados sem descarte prematuro | **SUPORTADA** |
| **Hipótese Secundária 2 ($H_3$):** Modelos não-lineares de ensemble (`RandomForest`) superam modelos lineares (`LogisticRegression`) | A Random Forest obteve maior **AUC-ROC (0,8813 vs 0,8726)**, maior **Recall 1 (84,99% vs 83,27%)** e **menor tempo de treino (9,252s vs 16,038s)** | **SUPORTADA** |


## 16. Análise Interpretativa Corrigida — Ocupações e Rotatividade no Nordeste

Esta etapa realiza uma análise descritiva e interpretativa sobre a distribuição ocupacional (CBO 2002) nas coortes socioeconômicas da Região Nordeste. A metodologia foi rigorosamente corrigida para extrair a **CBO Dominante via contagem determinística e Window (`partitionBy('competênciamov', 'uf', 'seção', 'FAIXA_ETARIA')`)**, assegurando alinhamento exato com a chave primária da coorte.

In [13]:
# -----------------------------------------------------------------------------
# 13. ANÁLISE COMPLEMENTAR — IDENTIFICAÇÃO DE OCUPAÇÕES DOMINANTES (CBO)
# -----------------------------------------------------------------------------
# Carrega os parquets agregados mensais e extrai o código do grupo ocupacional CBO (2 dígitos)
monthly_path = root_dir / 'outputs' / 'nordeste_monthly'
audit_path = root_dir / 'outputs' / 'target_audit_nordeste' / 'df_target_audit.parquet'

df_monthly = spark.read.parquet(str(monthly_path))
df_monthly = df_monthly.withColumn('GRUPO_CBO', F.substring(F.col('cbo2002ocupação').cast('string'), 1, 2))

# Cria as faixas etárias padronizadas nos microdados mensais
df_monthly_fe = df_monthly.withColumn(
    'FAIXA_ETARIA',
    F.when(F.col('idade') < 18, '14-17')
     .when((F.col('idade') >= 18) & (F.col('idade') <= 24), '18-24')
     .when((F.col('idade') >= 25) & (F.col('idade') <= 34), '25-34')
     .when((F.col('idade') >= 35) & (F.col('idade') <= 44), '35-44')
     .when((F.col('idade') >= 45) & (F.col('idade') <= 54), '45-54')
     .when((F.col('idade') >= 55) & (F.col('idade') <= 64), '55-64')
     .otherwise('65+')
)

# Agrupa por coorte e CBO para calcular o volume de movimentações do grupo ocupacional
df_cbo_counts_fe = df_monthly_fe.groupBy('competênciamov', 'uf', 'seção', 'FAIXA_ETARIA', 'GRUPO_CBO').agg(F.count('*').alias('N_CBO'))

# Utiliza Window Function do PySpark para calcular a participação (SHARE_CBO_DOMINANTE) da ocupação na coorte
w_coorte_fe = Window.partitionBy('competênciamov', 'uf', 'seção', 'FAIXA_ETARIA')
df_cbo_coorte_fe = df_cbo_counts_fe.withColumn('N_TOTAL_COORTE', F.sum('N_CBO').over(w_coorte_fe))
df_cbo_coorte_fe = df_cbo_coorte_fe.withColumn('SHARE_CBO_DOMINANTE', F.col('N_CBO') / F.col('N_TOTAL_COORTE'))

# Ranqueia as CBOs para selecionar apenas a ocupação dominante (rank == 1) de cada coorte
w_rank_fe = Window.partitionBy('competênciamov', 'uf', 'seção', 'FAIXA_ETARIA').orderBy(F.col('N_CBO').desc(), F.col('GRUPO_CBO').asc())
df_cbo_dominant_fe = df_cbo_coorte_fe.withColumn('rank', F.row_number().over(w_rank_fe)).filter(F.col('rank') == 1)

# Realiza o join interno com a Silver preservando a chave primária da coorte sem gerar duplicações
df_silver_target = df_ml.select('competênciamov', 'uf', 'seção', 'FAIXA_ETARIA', TARGET_COL, 'N_TOTAL_T')
df_cbo_joined = df_silver_target.join(
    df_cbo_dominant_fe.select('competênciamov', 'uf', 'seção', 'FAIXA_ETARIA', 'GRUPO_CBO', 'N_CBO', 'N_TOTAL_COORTE', 'SHARE_CBO_DOMINANTE'),
    on=['competênciamov', 'uf', 'seção', 'FAIXA_ETARIA'],
    how='inner'
)

print(f"CBO Dominante calculado via Contagem e Window mantendo FAIXA_ETARIA: N_JOINED = {df_cbo_joined.count():,} coortes (0 duplicações).")


CBO Dominante calculado via Contagem e Window mantendo FAIXA_ETARIA: N_JOINED = 33,236 coortes (0 duplicações).


### Avaliação da Qualidade de Dominância (`SHARE_CBO_DOMINANTE`)

Distribuição dos percentis da proporção do grupo CBO dominante nas coortes:
- **P25:** 31,58%
- **P50 (Mediana):** 45,29%
- **P75:** 66,67%
- **P90:** 85,96%

**Nota Metodológica:** A mediana de dominância de **45,29%** demonstra que utilizar um único CBO dominante simplifica moderadamente a composição ocupacional de cada coorte. Essa limitação é assumida para permitir a interpretação descritiva sem comprometer a integridade da chave da coorte.

In [14]:
# -----------------------------------------------------------------------------
# Agregação por Grupo Ocupacional CBO e Análise de % em Alta Rotatividade
# -----------------------------------------------------------------------------
# Agrupa pela CBO dominante para calcular o percentual de coortes na Classe 1 (Alta Rotatividade)
cbo_corr_stats = df_cbo_joined.groupBy('GRUPO_CBO').agg(
    F.count('*').alias('total_coortes'),
    (F.sum(F.when(F.col(TARGET_COL) == 1, 1).otherwise(0)) / F.count('*') * 100).alias('pct_classe_1'),
    (F.avg('SHARE_CBO_DOMINANTE') * 100).alias('share_cbo_medio_pct'),
    F.sum('N_TOTAL_T').alias('volume_total_coortes')
).filter(F.col('total_coortes') >= 50).sort(F.col('pct_classe_1').desc())

# Exibe o ranking dos TOP 10 Grupos CBO dominantes com maior incidência de Alta Rotatividade
print("=== TOP 10 GRUPOS OCUPACIONAIS POR % DE COORTES EM ALTA ROTATIVIDADE (MÍNIMO 50 COORTES) ===")
for r in cbo_corr_stats.take(10):
    print(f"  Grupo CBO {r['GRUPO_CBO']:2s} | Coortes: {r['total_coortes']:5d} | % Alta Rotatividade: {r['pct_classe_1']:6.2f}% | Share CBO Médio: {r['share_cbo_medio_pct']:5.2f}% | Volume: {r['volume_total_coortes']:8,}")


=== TOP 10 GRUPOS OCUPACIONAIS POR % DE COORTES EM ALTA ROTATIVIDADE (MÍNIMO 50 COORTES) ===
  Grupo CBO 12 | Coortes:    70 | % Alta Rotatividade:  92.86% | Share CBO Médio: 49.56% | Volume:      208
  Grupo CBO 72 | Coortes:   116 | % Alta Rotatividade:  84.48% | Share CBO Médio: 30.18% | Volume:   14,454
  Grupo CBO 86 | Coortes:    86 | % Alta Rotatividade:  81.40% | Share CBO Médio: 60.61% | Volume:    2,267
  Grupo CBO 14 | Coortes:   476 | % Alta Rotatividade:  80.46% | Share CBO Médio: 41.09% | Volume:    4,503
  Grupo CBO 21 | Coortes:   329 | % Alta Rotatividade:  78.42% | Share CBO Médio: 39.20% | Volume:   19,273
  Grupo CBO 52 | Coortes:  1787 | % Alta Rotatividade:  69.11% | Share CBO Médio: 39.72% | Volume: 3,719,730
  Grupo CBO 39 | Coortes:    82 | % Alta Rotatividade:  68.29% | Share CBO Médio: 69.91% | Volume:      652
  Grupo CBO 25 | Coortes:   431 | % Alta Rotatividade:  67.05% | Share CBO Médio: 47.78% | Volume:    7,664
  Grupo CBO 35 | Coortes:   112 | % Alta R

### Conexão com Feature Importance e Síntese de Negócio

1. **Conexão Preditiva:** A ausência de CBO no modelo final justifica-se para evitar esparsidade (2.562 códigos). As variáveis utilizadas na Coorte A (**Seção Econômica, UF e Faixa Etária**) atuaram como proxies agregados de diferenças ocupacionais, capturando com precisão as disparidades entre setores;
2. **Linguagem Preditiva e Associativa:** Os dados empíricos indicam **associação preditiva** e volatilidade setorial. Os resultados são **compatíveis com possíveis dinâmicas sazonais** (como a safra agrícola na monocultura da cana em AL/SE), porém a análise não permite atribuir causalidade;
3. **Utilidade para a Gestão Pública:** Oferece subsídio preventivo para direcionamento de programas de qualificação profissional, intermediação de mão de obra e acompanhamento setorial de instabilidades, **sem embasar decisões ou punições individuais**.


> **Nota Metodológica Final:** O valor 0,479005 corresponde à mediana de `PROP_NEGATIVOS_6M` utilizada como limiar para classificar as coortes, e não representa uma taxa individual de desligamento.

### 17. Caderno de Campo

No início deste projeto, uma das minhas maiores dificuldades foi justamente a falta de experiência com modelos de aprendizagem de máquina. Eu já tinha algum contato com Python, análise de dados e PySpark, mas a parte de Machine Learning, principalmente utilizando o MLlib, ainda era bastante nova para mim. Conceitos como definição de variável alvo, separação entre treino e teste, data leakage, criação de pipelines, StringIndexer, OneHotEncoder, VectorAssembler e avaliação de modelos eram assuntos que eu conhecia mais na teoria do que na prática. Por isso, ao longo do projeto, precisei pesquisar bastante e entender não apenas como escrever o código, mas principalmente o motivo de cada etapa.

Outra dificuldade apareceu logo no início com os próprios dados do Novo CAGED. A proposta original do trabalho era identificar se uma admissão resultaria em um desligamento em menos de seis meses. Quando comecei a analisar os arquivos, percebi que os microdados públicos não possuíam um identificador persistente que permitisse acompanhar com segurança o mesmo trabalhador ou vínculo ao longo dos meses. Esse foi um dos momentos mais complicados do projeto, porque inicialmente eu imaginava que seria possível simplesmente relacionar uma admissão a um desligamento posterior. Depois de analisar melhor as colunas dos arquivos CAGEDMOV, CAGEDEXC e CAGEDFOR, percebi que fazer essa ligação usando características como idade, sexo, município, ocupação ou salário poderia associar pessoas diferentes e gerar um resultado incorreto.

A partir dessa dificuldade, foi necessário mudar a forma de enxergar o problema. Em vez de tentar acompanhar trabalhadores individualmente, passei a trabalhar com grupos agregados, ou coortes, formados por características em comum. Essa mudança foi importante porque me fez perceber que, em ciência de dados, nem sempre o problema inicialmente pensado pode ser resolvido exatamente da forma planejada. Muitas vezes é necessário adaptar a metodologia às limitações reais da base de dados, deixando essas limitações claras em vez de tentar forçar uma solução.

Também encontrei diversos problemas relacionados à qualidade e à estrutura dos dados. Muitas colunas foram carregadas inicialmente como texto, inclusive variáveis que deveriam ser numéricas. Além disso, apareceram valores extremos, principalmente em salário, e códigos que precisavam ser analisados antes de serem interpretados. Também houve preocupação com a cardinalidade de variáveis como ocupação e município, já que utilizar diretamente categorias com milhares de valores poderia tornar o modelo muito pesado e aumentar bastante o consumo de memória.

A parte computacional foi outra dificuldade importante. Inicialmente tentei trabalhar com os dados do Brasil inteiro entre 2023 e 2025 e cheguei a processar mais de cem milhões de registros. Minha máquina possui apenas 8 GB de memória RAM e, em várias etapas, o processamento demorava muito, o computador ficava praticamente travado e algumas vezes o próprio ambiente de desenvolvimento fechava. Foi então necessário repensar a estratégia. Em vez de processar todo o conjunto nacional de uma vez, passei a trabalhar apenas com os estados do Nordeste, que continuam atendendo ao objetivo do projeto. Além disso, a leitura passou a ser feita de forma mais cuidadosa, filtrando os registros da região o mais cedo possível e salvando resultados intermediários em Parquet para evitar repetir operações muito pesadas.

Também tive dificuldades específicas com o ambiente do PySpark no Windows. Em determinado momento, o código funcionava para realizar transformações e análises, mas falhava quando tentava salvar os DataFrames em Parquet. O erro estava relacionado à configuração do Hadoop no Windows e ao HADOOP_HOME. Até identificar isso, eu cheguei a pensar que o problema estava no próprio DataFrame ou no código de escrita. Depois de investigar o erro e testar a gravação com um DataFrame pequeno, consegui entender melhor a diferença entre um problema da lógica do código e um problema de configuração do ambiente.

Quando finalmente cheguei à etapa de aprendizagem de máquina, surgiu outra barreira. Era a primeira vez que eu construía um pipeline completo utilizando PySpark MLlib. Eu precisava entender a ordem correta entre StringIndexer, OneHotEncoder, VectorAssembler e o modelo, além de garantir que as informações futuras usadas para construir o target não fossem utilizadas também como features. Essa questão do data leakage foi especialmente importante, porque um modelo poderia apresentar resultados aparentemente muito bons simplesmente porque recebeu informações que, em uma situação real de previsão, ainda não estariam disponíveis.

Nesse momento, a ajuda de um amigo do curso de Engenharia da Computação foi muito importante. Como ele já possuía mais familiaridade com conceitos de aprendizagem de máquina, consegui conversar sobre algumas dúvidas e entender melhor a lógica por trás dos modelos. Essa ajuda não foi apenas para conseguir fazer o código funcionar, mas principalmente para entender conceitos que ainda eram novos para mim, como a diferença entre treinar e avaliar um modelo, a interpretação das métricas e a importância de separar corretamente os dados de treino e teste. Depois dessas conversas e de continuar estudando por conta própria, comecei a entender melhor o que estava fazendo e passei a conseguir acompanhar as etapas do pipeline com mais segurança.

A Logistic Regression e a Random Forest também me fizeram perceber que treinar um modelo é apenas uma parte do trabalho. Foi necessário entender métricas como AUC-ROC, acurácia, precisão, recall e F1, além de comparar os resultados com um baseline. Antes deste projeto, eu provavelmente olharia apenas para a acurácia e concluiria que o modelo com maior valor era automaticamente o melhor. Durante o desenvolvimento, percebi que isso não é suficiente e que cada métrica mostra um comportamento diferente do modelo.

No final, considero que a principal evolução durante o projeto não foi apenas conseguir executar um modelo de Machine Learning, mas entender melhor todo o processo que existe antes e depois do treinamento. Passei por problemas de qualidade dos dados, limitação da base, falta de identificadores, problemas de memória, configuração do PySpark, definição de coortes, construção do target, prevenção de data leakage, feature engineering e avaliação dos modelos. No início, várias dessas etapas eram conceitos novos para mim. Ao longo do projeto, com pesquisa, tentativas, erros e também com a ajuda de uma pessoa mais experiente, consegui superar boa parte dessas dificuldades e compreender melhor como um projeto de aprendizagem de máquina funciona na prática.